# Consolidating collected reviews  and to see which need update  and which are entirely missing

### Check the places info parser to include border places

In [ ]:
import pandas as pd     
import numpy as np

In [ ]:
all_pools = pd.read_parquet('/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/germany_places.parquet')
all_places = all_pools.drop_duplicates(subset=['unique_identifier'])
all_pools.info()

In [ ]:
all_pools.shape

In [ ]:
### DO NOT RERUN THIS CELL
bd_pools = pd.read_csv('/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/all_inclusive_possible_pools_to_use_in_project_09042026.csv')

bd_pools.info()

In [ ]:
bd_pools.shape

In [ ]:
# Split bd_pools into two groups for matching
bd_has_uid = bd_pools[bd_pools['unique_identifier'].notna()]  # match on unique_identifier
bd_no_uid = bd_pools[bd_pools['unique_identifier'].isna() & bd_pools['place_id'].notna()]  # match on place_id only

# Merge 1: match on unique_identifier
merge1 = all_pools.merge(
    bd_has_uid[['unique_identifier', 'pool_id', 'pool_name']],
    on='unique_identifier',
    how='left'
)
merge2 =all_pools.merge(
    bd_no_uid[['place_id', 'pool_id', 'pool_name']],
    on='place_id',
    how='left'
)



# Recombine
all_pools_augmented = pd.concat([merge1, merge2]).drop_duplicates('unique_identifier').reset_index(drop=True)
all_pools_augmented.info()# Merge 2: for rows that didn't get a pool_id yet, try matching on place_id


In [ ]:
bd_pools[~bd_pools['pool_name'].isin(all_pools_augmented['pool_name'])]#.to_csv('/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/germany_pools_missed.csv', index=False)

In [ ]:
all_pools_augmented[all_pools_augmented['unique_identifier']== "0x4795b686b3aa2021:0x8418f815271cfe15"]  

In [ ]:
# check for duplicate pool_id and pool_name in bd_pools
bpd = bd_pools[(bd_pools['pool_name'].notna()) & (bd_pools['pool_id'].notna())]
## check for duplicate pool_id and pool_name in bdp
bpd[bpd.duplicated(subset=['pool_name'], keep=False)].sort_values(by=['pool_id', 'pool_name'])


In [ ]:
# def flatten_value(val):
#     """Flatten a value, handling both scalars and lists."""
#     if val is None:
#         return []
#     elif isinstance(val, list):
#         return val
#     else:
#         return [val]

# def merge_values_to_list(series):
#     """Convert a Series to a list of unique non-null values, or keep single value if only one.
#     Handles cases where values might already be lists from previous steps."""
#     # Flatten all values (scalars or lists)
#     flattened = []
#     for val in series:
#         flattened.extend(flatten_value(val))
    
#     # Get unique values
#     unique_vals = list(dict.fromkeys(flattened))  # Preserves order while removing duplicates
    
#     if len(unique_vals) == 0:
#         return None
#     elif len(unique_vals) == 1:
#         return unique_vals[0]
#     else:
#         return unique_vals

# def deduplicate_step1(df):
#     """
#     Step 1: Deduplicate on (unique_identifier, place_id).
#     For rows with same (unique_identifier, place_id):
#     - If pool_id and pool_name differ, merge them into lists
#     - Otherwise keep single values
#     """
#     df = df.copy()
    
#     grouped = df.groupby(['unique_identifier', 'place_id'], dropna=False)
    
#     results = []
#     for (uid, pid), group in grouped:
#         row = {
#             'unique_identifier': uid,
#             'place_id': pid,
#             'pool_id': merge_values_to_list(group['pool_id']),
#             'pool_name': merge_values_to_list(group['pool_name'])
#         }
#         results.append(row)
    
#     return pd.DataFrame(results)

# def deduplicate_step2(df):
#     """
#     Step 2: Deduplicate on unique_identifier only.
#     For rows with same non-null unique_identifier:
#     - If pool_id and pool_name differ, merge them into lists
#     - For place_id: pick one non-null value (or null if all null)

#     Rows with null unique_identifier are left unchanged (not deduplicated).
#     """
#     df = df.copy()

#     def pick_non_null(series):
#         """Pick first non-null value or return None"""
#         non_null = series.dropna()
#         return non_null.iloc[0] if len(non_null) > 0 else None

#     # Keep null unique_identifier rows untouched
#     null_uid_rows = df[df['unique_identifier'].isna()].copy()

#     # Deduplicate only non-null unique_identifier rows
#     non_null_uid = df[df['unique_identifier'].notna()].copy()
#     grouped = non_null_uid.groupby('unique_identifier', dropna=False)

#     results = []
#     for uid, group in grouped:
#         row = {
#             'unique_identifier': uid,
#             'pool_id': merge_values_to_list(group['pool_id']),
#             'pool_name': merge_values_to_list(group['pool_name']),
#             'place_id': pick_non_null(group['place_id'])
#         }
#         results.append(row)

#     dedup_non_null = pd.DataFrame(results)

#     # Combine deduplicated non-null rows with untouched null rows
#     combined = pd.concat([dedup_non_null, null_uid_rows], ignore_index=True)
#     return combined

# def deduplicate_step3(df):
#     df = df.copy()

#     def pick_non_null(series):
#         non_null = series.dropna()
#         return non_null.iloc[0] if len(non_null) > 0 else None

#     # Exempt null place_id rows — same pattern as Step 2
#     null_pid_rows = df[df['place_id'].isna()].copy()
#     non_null_pid = df[df['place_id'].notna()].copy()

#     grouped = non_null_pid.groupby('place_id', dropna=False)

#     results = []
#     for pid, group in grouped:
#         row = {
#             'place_id': pid,
#             'pool_id': merge_values_to_list(group['pool_id']),
#             'pool_name': merge_values_to_list(group['pool_name']),
#             'unique_identifier': pick_non_null(group['unique_identifier'])
#         }
#         results.append(row)

#     dedup_non_null = pd.DataFrame(results)
#     return pd.concat([dedup_non_null, null_pid_rows], ignore_index=True)

# # Execute the three deduplication steps
# print("Original bd_pools shape:", bd_pools.shape)
# print("\n" + "="*60)
# print("STEP 1: Deduplicate on (unique_identifier, place_id)")
# print("="*60)
# bd_pools_step1 = deduplicate_step1(bd_pools)
# print(f"After step 1 shape: {bd_pools_step1.shape}")
# print(f"Rows with list in pool_id: {bd_pools_step1['pool_id'].apply(lambda x: isinstance(x, list)).sum()}")
# print(f"Rows with list in pool_name: {bd_pools_step1['pool_name'].apply(lambda x: isinstance(x, list)).sum()}")

# print("\n" + "="*60)
# print("STEP 2: Deduplicate on unique_identifier")
# print("="*60)
# bd_pools_step2 = deduplicate_step2(bd_pools_step1)
# print(f"After step 2 shape: {bd_pools_step2.shape}")
# print(f"Rows with list in pool_id: {bd_pools_step2['pool_id'].apply(lambda x: isinstance(x, list)).sum()}")
# print(f"Rows with list in pool_name: {bd_pools_step2['pool_name'].apply(lambda x: isinstance(x, list)).sum()}")
# print(f"Null place_id values: {bd_pools_step2['place_id'].isna().sum()}")
# print(f"Null unique_identifier values kept: {bd_pools_step2['unique_identifier'].isna().sum()}")

# #
# print("\n" + "="*60)
# print("STEP 3: Deduplicate on place_id")
# print("="*60)
# bd_pools_final = deduplicate_step3(bd_pools_step2)
# print(f"After step 3 shape: {bd_pools_final.shape}")
# print(f"Rows with list in pool_id: {bd_pools_final['pool_id'].apply(lambda x: isinstance(x, list)).sum()}")
# print(f"Rows with list in pool_name: {bd_pools_final['pool_name'].apply(lambda x: isinstance(x, list)).sum()}")
# print(f"Null place_id values kept: {bd_pools_final['place_id'].isna().sum()}")
# print(f"Null unique_identifier values kept: {bd_pools_final['unique_identifier'].isna().sum()}")    

# # Show a sample of merged rows
# print("\n" + "="*60)
# print("SAMPLE OF MERGED ROWS (with lists)")
# print("="*60)
# merged_rows = bd_pools_final[
#     (bd_pools_final['pool_id'].apply(lambda x: isinstance(x, list))) |
#     (bd_pools_final['pool_name'].apply(lambda x: isinstance(x, list)))
# ]
# print(f"Total merged rows: {len(merged_rows)}")
# print(merged_rows.head(10))

In [ ]:
# # Quick summary
# reduction = len(bd_pools) - len(bd_pools_final)
# reduction_pct = 100 * reduction / len(bd_pools)

# print(f"FINAL DEDUPLICATION RESULTS:")
# print(f"  Original rows:        {len(bd_pools)}")
# print(f"  Final rows:           {len(bd_pools_final)}")
# print(f"  Rows reduced:         {reduction} ({reduction_pct:.1f}%)")
# print(f"\nMerged values in final dataset:")
# print(f"  pool_id as lists:     {bd_pools_final['pool_id'].apply(lambda x: isinstance(x, list)).sum()}")
# print(f"  pool_name as lists:   {bd_pools_final['pool_name'].apply(lambda x: isinstance(x, list)).sum()}")
# print(f"\nNull values in final dataset:")
# print(f"  unique_identifier:    {bd_pools_final['unique_identifier'].isna().sum()}")
# print(f"  place_id:             {bd_pools_final['place_id'].isna().sum()}")
# print(f"  pool_id:              {bd_pools_final['pool_id'].isna().sum()}")
# print(f"  pool_name:            {bd_pools_final['pool_name'].isna().sum()}")

# # Show one example of merged values
# example = bd_pools_final[bd_pools_final['pool_id'].apply(lambda x: isinstance(x, list))].iloc[0]
# print(f"\nExample merged row:")
# print(f"  unique_identifier: {example['unique_identifier']}")
# print(f"  place_id: {example['place_id']}")
# print(f"  pool_id: {example['pool_id']}")
# print(f"  pool_name: {example['pool_name']}")

In [ ]:
# def count_values(val):
#     """Count the number of values: lists return length, scalars return 1, null returns 0"""
#     if val is None:
#         return 0
#     elif isinstance(val, list):
#         return len(val)
#     else:
#         return 1

# # Add tracking columns
# bd_pools_final['pool_id_count'] = bd_pools_final['pool_id'].apply(count_values)
# bd_pools_final['pool_name_count'] = bd_pools_final['pool_name'].apply(count_values)

# print("Added columns 'pool_id_count' and 'pool_name_count'")
# print(f"\npool_id_count distribution:")
# print(bd_pools_final['pool_id_count'].value_counts().sort_index())
# print(f"\npool_name_count distribution:")
# print(bd_pools_final['pool_name_count'].value_counts().sort_index())

# print(f"\nSample rows with different counts:")
# print(bd_pools_final[['unique_identifier', 'place_id', 'pool_id_count', 'pool_name_count', 'pool_id', 'pool_name']].head(10))

In [ ]:
# # Diagnose why flattened unique pool_id > pool_name


# def _is_missing(x):
#     return x is None or (not isinstance(x, list) and pd.isna(x))


# def _to_list(x):
#     if isinstance(x, list):
#         vals = []
#         for v in x:
#             if not _is_missing(v) and v not in vals:
#                 vals.append(v)
#         return vals
#     return [] if _is_missing(x) else [x]


# pairs = []
# for _, r in bd_pools_final[['pool_name', 'pool_id']].iterrows():
#     names = _to_list(r['pool_name'])
#     ids = _to_list(r['pool_id'])

#     # row-level cartesian pairs of the merged sets
#     for n in names:
#         for pid in ids:
#             pairs.append((n, pid))

# pairs_df = pd.DataFrame(pairs, columns=['pool_name', 'pool_id']).drop_duplicates()

# name_to_id_counts = pairs_df.groupby('pool_name')['pool_id'].nunique().sort_values(ascending=False)
# id_to_name_counts = pairs_df.groupby('pool_id')['pool_name'].nunique().sort_values(ascending=False)

# print('Flattened unique values:')
# print('pool_name:', bd_pools_final['pool_name'].apply(_to_list).explode().nunique())
# print('pool_id:', bd_pools_final['pool_id'].apply(_to_list).explode().nunique())
# print()

# print('How many names map to multiple IDs:', int((name_to_id_counts > 1).sum()))
# print('How many IDs map to multiple names:', int((id_to_name_counts > 1).sum()))
# print()

# print('Top names with multiple IDs:')
# print(name_to_id_counts[name_to_id_counts > 1].head(20))
# print()

# print('Top IDs with multiple names:')
# print(id_to_name_counts[id_to_name_counts > 1].head(20))

# # Direct reason for pool_id > pool_name
# extra_ids_vs_names = bd_pools_final['pool_id'].apply(_to_list).explode().nunique() - bd_pools_final['pool_name'].apply(_to_list).explode().nunique()
# print()
# print('Difference (unique pool_id - unique pool_name):', int(extra_ids_vs_names))

In [ ]:
# # Check row-level mismatch between counts
# mismatch = bd_pools_final[bd_pools_final['pool_id_count'] != bd_pools_final['pool_name_count']].copy()

# print('Rows with pool_id_count != pool_name_count:', len(mismatch))
# if len(mismatch) > 0:
#     mismatch['abs_diff'] = (mismatch['pool_id_count'] - mismatch['pool_name_count']).abs()
#     print('\nLargest mismatches:')
#     print(
#         mismatch.sort_values('abs_diff', ascending=False)[
#             ['unique_identifier', 'place_id', 'pool_id_count', 'pool_name_count', 'pool_id', 'pool_name']
#         ].head(10)
#     )

# print('\nRows with IDs but no names:', int(((bd_pools_final['pool_id_count'] > 0) & (bd_pools_final['pool_name_count'] == 0)).sum()))
# print('Rows with names but no IDs:', int(((bd_pools_final['pool_name_count'] > 0) & (bd_pools_final['pool_id_count'] == 0)).sum()))

In [ ]:
# # ============================================================
# # INVESTIGATION 1: The 5 extra pool_ids (no matching pool_name)
# # ============================================================
# print("="*60)
# print("INVESTIGATION 1: 5 extra pool_ids vs pool_names")
# print("="*60)

# def _is_missing(x):
#     return x is None or (not isinstance(x, list) and pd.isna(x))

# def _to_list(x):
#     if isinstance(x, list):
#         vals = []
#         for v in x:
#             if not _is_missing(v) and v not in vals:
#                 vals.append(v)
#         return vals
#     return [] if _is_missing(x) else [x]

# # Explode all pool_ids and pool_names flat
# all_ids = (
#     bd_pools_final['pool_id'].apply(_to_list)
#     .explode()
#     .dropna()
#     .unique()
# )
# all_names = (
#     bd_pools_final['pool_name'].apply(_to_list)
#     .explode()
#     .dropna()
#     .unique()
# )

# print(f"Unique pool_ids: {len(all_ids)}")
# print(f"Unique pool_names: {len(all_names)}")

# # Find rows where pool_id and pool_name counts differ
# count_mismatch = bd_pools_final[
#     bd_pools_final['pool_id_count'] != bd_pools_final['pool_name_count']
# ]
# print(f"\nRows where pool_id_count != pool_name_count: {len(count_mismatch)}")
# display(count_mismatch[['unique_identifier', 'place_id', 'pool_id', 'pool_name', 'pool_id_count', 'pool_name_count']])


# # ============================================================
# # INVESTIGATION 2: Many-to-many pool_id <-> pool_name mapping
# # ============================================================
# print("\n" + "="*60)
# print("INVESTIGATION 2: pool_id <-> pool_name many-to-many")
# print("="*60)

# # Build flat (pool_name, pool_id) pairs
# pairs = []
# for _, r in bd_pools_final[['pool_name', 'pool_id', 'unique_identifier', 'place_id']].iterrows():
#     names = _to_list(r['pool_name'])
#     ids   = _to_list(r['pool_id'])
#     for n in names:
#         for pid in ids:
#             pairs.append((n, pid, r['unique_identifier'], r['place_id']))

# pairs_df = pd.DataFrame(pairs, columns=['pool_name', 'pool_id', 'unique_identifier', 'place_id']).drop_duplicates()

# # Names mapping to multiple IDs
# name_to_ids = pairs_df.groupby('pool_name')['pool_id'].nunique().sort_values(ascending=False)
# multi_name  = name_to_ids[name_to_ids > 1]

# # IDs mapping to multiple names
# id_to_names = pairs_df.groupby('pool_id')['pool_name'].nunique().sort_values(ascending=False)
# multi_id    = id_to_names[id_to_names > 1]

# print(f"pool_names mapping to multiple pool_ids: {len(multi_name)}")
# print(f"pool_ids mapping to multiple pool_names: {len(multi_id)}")

# print("\nTop pool_names with multiple pool_ids:")
# print(multi_name.head(10))
# print("\nTop pool_ids with multiple pool_names:")
# print(multi_id.head(10))

# # For the worst offenders, show which rows they appear in
# print("\nRows containing the top multi-id pool_name:")
# worst_name = multi_name.index[0]
# print(f"  pool_name: '{worst_name}'")
# display(
#     pairs_df[pairs_df['pool_name'] == worst_name]
#     [['pool_name', 'pool_id', 'unique_identifier', 'place_id']]
# )

# print("\nRows containing the top multi-name pool_id:")
# worst_id = multi_id.index[0]
# print(f"  pool_id: {worst_id}")
# print(
#     pairs_df[pairs_df['pool_id'] == worst_id]
#     [['pool_name', 'pool_id', 'unique_identifier', 'place_id']]
# )


# # ============================================================
# # INVESTIGATION 3: Multiple (pool_id, pool_name) pairs per unique_identifier
# # ============================================================
# print("\n" + "="*60)
# print("INVESTIGATION 3: Multiple pool pairs per unique_identifier")
# print("="*60)

# uid_pair_counts = (
#     pairs_df[pairs_df['unique_identifier'].notna()]
#     .groupby('unique_identifier')
#     .apply(lambda g: len(g[['pool_id', 'pool_name']].drop_duplicates()))
#     .sort_values(ascending=False)
# )

# print(f"unique_identifiers with >1 (pool_id, pool_name) pair: {(uid_pair_counts > 1).sum()}")
# print("\nDistribution of pair counts per unique_identifier:")
# print(uid_pair_counts.value_counts().sort_index())

# print("\nTop unique_identifiers with most pairs:")
# print(uid_pair_counts.head(10))

# # Show the actual pairs for the worst offender
# worst_uid = uid_pair_counts.index[0]
# print(f"\nAll pairs for worst unique_identifier ({worst_uid}):")
# display(
#     pairs_df[pairs_df['unique_identifier'] == worst_uid]
#     [['pool_id', 'pool_name', 'place_id']]
#     .sort_values('pool_id')
# )

In [ ]:
#bd_pools_final[['place_id', 'unique_identifier', 'pool_id', 'pool_name', 'pool_name_count', 'pool_id_count']].to_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/all_inclusive_possible_pools_to_use_in_project_09042026.csv", index=False)

## Check that the places to collect reviews for are actually pools

In [ ]:
list_place_types = ['aquatic center', 'indoor swimming pool', 'lake', 'lake shore swimming area','lido',
                    'outdoor bath', 'outdoor swimming pool', 'public bath', 'public bathroom', 
                    'public sauna', 'public beach','public swimming pool', 'recreation center', 'spa','spa and health club',
                    'sports complex', 'swimming facility', 'swimming lake','swimming pool', 'thermal baths',
                    'water', 'water park']

In [ ]:
#first  make the promary place_type lower case
all_pools["primary_place_type"] = all_pools["primary_place_type"].str.lower()
actual_pools = all_pools[all_pools["primary_place_type"].isin(list_place_types)]
actual_pools = actual_pools.drop_duplicates(subset=['unique_identifier']).reset_index(drop=True)
#dropo columns with all null values
allall_pools =all_pools.dropna(axis=1, how='all')
actual_pools.info()

In [ ]:
#do not rerun, updated version is in the  data_augmentation_13042026.ipynb
#actual_pools.to_parquet("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/processed_data/actual_pools.parquet", index=False)

In [ ]:
#check those that have  reviews
pools_with_reviews = actual_pools[actual_pools['total_reviews'].notna() & (actual_pools['total_reviews'] > 0)]
#dropo columns with all null values
pools_with_reviews = pools_with_reviews.dropna(axis=1, how='all')
pools_with_reviews.info()

In [ ]:
pools_with_reviews.to_parquet("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/processed_data/actual_pools_with_reviews.parquet")

In [ ]:
## create new reviews  file for reviews collection
pools_with_reviews['expected_jsons'] = pools_with_reviews['total_reviews'].apply(lambda x:np.ceil(x/10))
pools_with_reviews_ids = pools_with_reviews[['place_id', 'unique_identifier', 'total_reviews', 'expected_jsons']]
pools_with_reviews_ids

In [ ]:
#now we need to create the  collection jsons column
collection_reviews = pd.read_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/review_counts_with_latest_next_key_12042026.csv")
collection_reviews.info()   

In [ ]:
pools_with_reviews_ids = pools_with_reviews_ids.merge(
    collection_reviews,
    on=['place_id', 'unique_identifier'],
    how='left'
)

# Use collected_reviews if present, otherwise fall back to review_count
count_col = 'collected_reviews' if 'collected_reviews' in pools_with_reviews_ids.columns else 'review_count'

# Rule:
# 1..9 reviews  -> 1 json
# >=10 reviews  -> ceil(reviews / 10)
# <=0 or missing -> 0 json
pools_with_reviews_ids['collected_jsons'] = (
    pd.to_numeric(pools_with_reviews_ids[count_col], errors='coerce')
    .fillna(0)
    .apply(lambda x: int(np.ceil(x / 10)) if x > 0 else 0)
)

pools_with_reviews_ids[['place_id', 'unique_identifier','total_reviews', 'expected_jsons',count_col, 'collected_jsons', 'next_key']].head()


In [ ]:
pools_with_reviews_ids[['place_id', 'unique_identifier','total_reviews', 'expected_jsons', 'collected_jsons']].to_csv("/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/germany_places_within.csv")

## checking  if within germany

In [ ]:
pools_within_germany = pd.read_parquet('/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/germany_places_within.parquet')
pools_within_germany.info()

In [ ]:
pools_within_germany.head()

In [ ]:
#chceck the address and coordinates of the pools nots in the within file
pools_not_within = all_pools[
    (~all_pools['unique_identifier'].isin(pools_within_germany['unique_identifier']))
    & (all_pools['total_reviews'].notna())
].copy()
pools_not_within.info() 

In [ ]:
pools_not_within['expected_jsons'] = pools_not_within['total_reviews'].apply(lambda x: np.ceil(x/10))
pools_not_within[['place_name', 'address', 'latitude', 'longitude', 'total_reviews', 'expected_jsons']].head()

In [ ]:
# Check whether coordinates of pools_not_within fall inside Germany polygons
import geopandas as gpd

# Tune this if needed: points up to this distance from the border are treated as likely in Germany
BORDER_TOLERANCE_METERS = 5000

germany_shp = gpd.read_file(
    '/home/ethel/Desktop/swimming_pool_project/btw25_geometrie_wahlkreise_shp/btw25_geometrie_wahlkreise_shp.shp'
).to_crs('EPSG:4326')

# Build point geometries only for rows with valid coordinates
coords_df = pools_not_within.dropna(subset=['latitude', 'longitude']).copy()
coords_df = coords_df[
    coords_df['latitude'].between(-90, 90) & coords_df['longitude'].between(-180, 180)
]

points_gdf = gpd.GeoDataFrame(
    coords_df,
    geometry=gpd.points_from_xy(coords_df['longitude'], coords_df['latitude']),
    crs='EPSG:4326'
)

# Union of German polygons
germany_geom = germany_shp.geometry.union_all()

# Strict spatial check
strict_in_germany = points_gdf.geometry.within(germany_geom) | points_gdf.geometry.touches(germany_geom)

# Distance-to-border check in meters (equal-area CRS for Europe)
germany_3035 = gpd.GeoSeries([germany_geom], crs='EPSG:4326').to_crs('EPSG:3035').iloc[0]
points_3035 = points_gdf.to_crs('EPSG:3035')

# 0 for points inside polygon, >0 outside
distance_to_germany_m = points_3035.geometry.distance(germany_3035)
within_tolerance = distance_to_germany_m <= BORDER_TOLERANCE_METERS

# Final classification
points_gdf['strict_in_germany'] = strict_in_germany
points_gdf['distance_to_germany_m'] = distance_to_germany_m.round(1)
points_gdf['likely_in_germany'] = strict_in_germany | within_tolerance

print(f"Pools checked (with coordinates): {len(points_gdf)}")
print(f"Strictly inside Germany: {points_gdf['strict_in_germany'].sum()}")
print(f"Likely inside Germany (+{BORDER_TOLERANCE_METERS}m tolerance): {points_gdf['likely_in_germany'].sum()}")
print(f"Likely outside Germany: {(~points_gdf['likely_in_germany']).sum()}")

outside_germany = points_gdf.loc[~points_gdf['likely_in_germany'], [
    'unique_identifier', 'place_name', 'address', 'latitude', 'longitude', 'total_reviews',
    'distance_to_germany_m'
]].sort_values('distance_to_germany_m', ascending=True)

# If many are close to the border, they are probably in Germany with slightly noisy coordinates
borderline = points_gdf.loc[
    (~points_gdf['strict_in_germany']) & (points_gdf['distance_to_germany_m'] <= BORDER_TOLERANCE_METERS),
    ['unique_identifier', 'place_name', 'address', 'latitude', 'longitude', 'distance_to_germany_m']
]

print(f"Borderline points now considered likely in Germany: {len(borderline)}")

outside_germany.head(25)

In [ ]:
points_gdf

In [ ]:
# Add likely-in-Germany pools from pools_not_within into the within dataset
within_path = '/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/germany_places_within.parquet'
updated_within_path = '/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/germany_places_within_updated.parquet'

# Reuse result from previous cell
to_add_ids = points_gdf.loc[points_gdf['likely_in_germany'], 'unique_identifier']
pools_to_add = pools_not_within[pools_not_within['unique_identifier'].isin(to_add_ids)].copy()

print(f"Rows selected to add: {len(pools_to_add)}")

# Align columns and combine
common_cols = [c for c in pools_within_germany.columns if c in pools_to_add.columns]
pools_within_updated = pd.concat(
    [pools_within_germany[common_cols], pools_to_add[common_cols]],
    ignore_index=True
)

# Deduplicate after concat
before_dedup = len(pools_within_updated)
pools_within_updated = pools_within_updated.drop_duplicates(
    subset=['unique_identifier'], keep='first'
).copy()
after_dedup = len(pools_within_updated)

print(f"Rows before dedup: {before_dedup}")
print(f"Rows after dedup:  {after_dedup}")
print(f"Net rows added:    {after_dedup - len(pools_within_germany)}")

# Save updated file (safe: new output file)
pools_within_updated.to_parquet(updated_within_path, index=False)
print(f"Saved updated within file to: {updated_within_path}")

# Optional: overwrite original after verification
# pools_within_updated.to_parquet(within_path, index=False)

pools_within_updated.tail()

In [ ]:
#pools_within_updated.to_csv('/home/ethel/Desktop/swimming_pool_project/german_pools_data_09042026/germany/germany_places_within.csv', index=False)

In [ ]:
# Why shape changes with drop_duplicates on place_id
n_total = len(actual_pools)
valid_place_id = actual_pools['place_id'].notna() & (actual_pools['place_id'].astype(str).str.strip() != '')

n_valid = int(valid_place_id.sum())
n_invalid = int((~valid_place_id).sum())

# Dedup across all rows (includes null/blank place_id as a single repeated value)
all_dedup = actual_pools.drop_duplicates(subset=['place_id'])

# Dedup only valid place_id rows
valid_dedup = actual_pools.loc[valid_place_id].drop_duplicates(subset=['place_id'])

print('actual_pools rows:', n_total)
print('valid place_id rows:', n_valid)
print('null/blank place_id rows:', n_invalid)
print()
print('Rows after drop_duplicates on place_id (all rows):', len(all_dedup))
print('Rows after drop_duplicates on place_id (valid place_id only):', len(valid_dedup))
print()
print('Valid duplicate place_id rows:', int(actual_pools.loc[valid_place_id].duplicated(subset=['place_id'], keep=False).sum()))
print('Blank place_id unique values (repr):', actual_pools.loc[~valid_place_id, 'place_id'].map(repr).value_counts().head(10).to_dict())